## Preprocessing Text

In [ ]:
import pandas as pd
import re

class PaperPreprocessor:

    def clean_arxiv_id(self, url):
        match = re.search(r'(\d{4}\.\d+)', str(url))
        return match.group(1) if match else url

    def clean_text(self, text):

        if pd.isna(text):
            return ""

        text = str(text)

        text = re.sub(r'\\cite\{.*?\}', '', text)
        text = re.sub(r'\\ref\{.*?\}', '', text)
        text = re.sub(r'\\url\{.*?\}', '', text)
        text = re.sub(r'\s+', ' ', text)

        return text.strip()

    def build_doc_text(self, title, abstract):

        title = self.clean_text(title)
        abstract = self.clean_text(abstract)

        text = f"{title}. {title}. {abstract}"

        return text.strip()
    
    def preprocess_dataset(self, path, save_path=None):

        df = pd.read_parquet(path)

        df = df.rename(columns={"id": "doc_id"})

        df["doc_id"] = df["doc_id"].apply(self.clean_arxiv_id)

        title = df["title"].fillna("").astype(str)
        abstract = df["abstract"].fillna("").astype(str)

        title = title.str.replace(r'\\cite\{.*?\}', '', regex=True)
        title = title.str.replace(r'\\ref\{.*?\}', '', regex=True)
        title = title.str.replace(r'\\url\{.*?\}', '', regex=True)
        title = title.str.replace(r'\s+', ' ', regex=True)

        abstract = abstract.str.replace(r'\\cite\{.*?\}', '', regex=True)
        abstract = abstract.str.replace(r'\\ref\{.*?\}', '', regex=True)
        abstract = abstract.str.replace(r'\\url\{.*?\}', '', regex=True)
        abstract = abstract.str.replace(r'\s+', ' ', regex=True)

        df["text"] = title + ". " + title + ". " + abstract

        df = df[["doc_id", "text"]]
        df = df.drop_duplicates(subset="doc_id")
        if save_path:
            df.to_parquet(save_path, index=False)

        return df

## Embedding Text

In [ ]:
import faiss
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

class E5FaissRetriever:

    def __init__(self, model_name="intfloat/e5-base", device=None, cache_dir="../models"):

        self.device = device if device else (
            "cuda" if torch.cuda.is_available() else "cpu"
        )

        self.model = SentenceTransformer(model_name, 
                                         device=self.device,
                                         cache_folder=cache_dir)
        self.index = None
        self.doc_ids = None

    def encode_documents(self, texts, batch_size=128):

        texts = ["passage: " + t for t in texts]

        embeddings = []

        for i in tqdm(range(0, len(texts), batch_size), desc="Encoding docs"):
            batch = texts[i:i+batch_size]

            emb = self.model.encode(
                batch,
                convert_to_numpy=True,
                normalize_embeddings=True,
                show_progress_bar=False
            )

            embeddings.append(emb)

        return np.vstack(embeddings)

    def encode_queries(self, queries):

        queries = ["query: " + q for q in queries]

        emb = self.model.encode(
            queries,
            convert_to_numpy=True,
            normalize_embeddings=True
        )

        return emb

    def build_index(self, embeddings, doc_ids):

        embeddings = embeddings.astype("float32")

        dim = embeddings.shape[1]

        index = faiss.IndexHNSWFlat(dim, 32, faiss.METRIC_INNER_PRODUCT)

        index.hnsw.efConstruction = 200
        index.hnsw.efSearch = 128

        index.add(embeddings)

        self.index = index
        self.doc_ids = np.array(doc_ids)

    def search(self, query, top_k=10):

        q_emb = self.encode_queries([query])

        scores, indices = self.index.search(q_emb, top_k)

        results = self.doc_ids[indices[0]]

        return results, scores[0]

    def save(self, path):

        faiss.write_index(self.index, path)

    def load(self, path):

        self.index = faiss.read_index(path)

In [ ]:
import numpy as np

processor = PaperPreprocessor()
retriever = E5FaissRetriever()

df = processor.preprocess_dataset(
    "data/raw/papers_2015_2025.parquet", "data/preprocessed/emb_base.parquet"
)

print("Encoding documents...")

embeddings = retriever.encode_documents(
    df["text"].tolist()
)

print("Building FAISS index...")

retriever.build_index(
    embeddings,
    df["doc_id"].tolist()
)

retriever.save("data/preprocessed/paper_index.faiss")

np.save("data/preprocessed/doc_ids.npy", df["doc_id"].values)

print("Done.")